# EXP3: Dynamics Classification

## Centralized imports
Imports all required libraries. Run this cell first.

In [ ]:
# Centralized imports

import os
import sys
import json
import glob
import shutil
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import sparse
from skmultilearn.model_selection import IterativeStratification

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from transformers import (
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    PrinterCallback,
)
from transformers.utils import logging as hf_logging

import safetensors.torch as st

from IPython.display import display, Image as IPImage


## Cell 0 — Path Configuration
Central path registry for the entire notebook.
Every downstream cell reads paths from the `paths` object defined here — no hardcoded
directories anywhere else. Update **only this cell** when migrating between machines.


In [ ]:
# Cell 0 — Path Configuration

GROUPS = ['harmonic_intervals_loose', 'harmonic_intervals_strict', 'triads_loose', 'triads_strict', '7th_chords_loose', '7th_chords_strict']


def _discover_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "mini_secd_demo").is_dir() or (candidate / "experiments").is_dir():
            return candidate
    return start


REPO_ROOT = _discover_repo_root()
RUN_MODE = os.getenv("SECD_RUN_MODE", "demo" if (REPO_ROOT / "mini_secd_demo").is_dir() else "full").lower()
DEFAULT_BASE = REPO_ROOT / "mini_secd_demo" if RUN_MODE == "demo" and (REPO_ROOT / "mini_secd_demo").is_dir() else REPO_ROOT
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEMO_MAX_STEPS = 1 if RUN_MODE == "demo" else -1
DEMO_NUM_TRAIN_EPOCHS = 1 if RUN_MODE == "demo" else 40
DEMO_TRAIN_BATCH_SIZE = 2 if RUN_MODE == "demo" else 64
DEMO_EVAL_BATCH_SIZE = 4 if RUN_MODE == "demo" else 128
DEMO_NUM_WORKERS = 0 if RUN_MODE == "demo" or DEVICE.type == "cpu" else 2


@dataclass
class ExpPaths:
    base: str = os.getenv("SECD_BASE", str(DEFAULT_BASE))
    cache_root: str = ""
    outdir: str = ""
    split_dir: str = ""
    csv_dirs: list = field(default_factory=list)

    def __post_init__(self):
        self.base = str(Path(self.base).expanduser().resolve())
        self.cache_root = os.path.join(self.base, "mel_cache_ast16k_256")
        self.outdir = os.path.join(self.base, "EXP3_AST_DYN")
        self.split_dir = os.path.join(self.outdir, "exp3_splits")

        self.csv_dirs = []
        for group in GROUPS:
            d = os.path.join(self.base, group, "csv")
            if os.path.isdir(d):
                self.csv_dirs.append(d)

    def verify(self):
        errors = []
        if not os.path.isdir(self.cache_root):
            errors.append(f"Cache not found: {self.cache_root}")
        if not self.csv_dirs:
            errors.append("No CSV directories found")
        for d in self.csv_dirs:
            csvs = [f for f in os.listdir(d)
                    if f.endswith(".csv") and "-original" not in f]
            if not csvs:
                errors.append(f"No usable CSVs in {d}")
        if errors:
            for e in errors:
                print(f"  {e}")
            raise RuntimeError("Path verification failed")

        os.makedirs(self.outdir, exist_ok=True)
        os.makedirs(self.split_dir, exist_ok=True)

        all_csvs = []
        for d in self.csv_dirs:
            all_csvs += [os.path.join(d, f) for f in os.listdir(d)
                         if f.endswith(".csv") and "-original" not in f]

        print(f"  Run mode   : {RUN_MODE}")
        print(f"  Device     : {DEVICE}")
        print(f"  Base       : {self.base}")
        print(f"  Cache      : {self.cache_root}")
        print(f"  Output     : {self.outdir}")
        print(f"  Splits     : {self.split_dir}")
        print(f"  CSV dirs   : {len(self.csv_dirs)}")
        print(f"  CSV files  : {len(all_csvs)} (excluding -original)")
        return all_csvs


paths = ExpPaths()
all_csv_paths = paths.verify()
print("\nPath configuration OK")


## Cell 3 — Constants, Paths, CSV Inventory (EXP3: dynamics)
Establishes all experiment-level constants for EXP3.
EXP3 classifies **global dynamics level** (pianissimo / piano / mezzo-piano / forte / fortissimo).
Uses strict and loose subsets after applying the constraint.


In [ ]:
# Cell 3 — Constants, paths, CSV inventory

# ===================== Instruments =====================
INSTRUMENTS = ["cello", "viola", "violin2", "violin1"]
INSTR_ORDER = INSTRUMENTS
assert INSTR_ORDER == ["cello", "viola", "violin2", "violin1"]

# ===================== Label vocabularies (canonical) =====================
DYNAMICS = [
    "pianissimo", "piano", "mezzo-piano", "mezzo-forte",
    "forte", "fortissimo", "crescendo", "decrescendo",
] # 8 classes

TECHNIQUES = [
    "arco-normal",
    "arco-sul-ponticello",
    "arco-sul-tasto",
    "arco-tremolo",
    "arco-martele",
    "arco-glissando",
    "arco-au-talon",
    "arco-col-legno-tratto",
    "arco-col-legno-battuto",
    "arco-harmonic",
    "natural-harmonic",
    "artificial-harmonic",
    "arco-major-trill",
    "arco-minor-trill",
    "molto-vibrato",
    "non-vibrato",
    "pizz-normal",
    "pizz-tremolo",
    "pizz-glissando",
    "snap-pizz",
    "con-sord",
] # 21 classes

# ===================== Audio / Feature constants =====================
TARGET_SR  = 16000 # AST-native sample rate
TARGET_FRAMES = 256  # ~2.6 sec at 16 kHz / 10 ms hop (fits 2-sec clips)
N_MELS  = 128
DURATION  = 2.0  # clip length in seconds

# ===================== Reproducibility =====================
SEED = 1337

# ===================== Filesystem (from Cell 0) =====================
CACHE_ROOT = paths.cache_root
os.makedirs(paths.outdir, exist_ok=True)

# ===================== Per-instrument column helpers =====================
def find_audio_cols(df: pd.DataFrame):
    return [c for c in df.columns if c.endswith("_file")]

def assert_audio_cols(df: pd.DataFrame, csv_path: str):
    cols = find_audio_cols(df)
    if not cols:
        raise ValueError(
            f"No *_file columns found in {csv_path}. "
            f"Expected per-instrument columns such as 'cello_file', 'viola_file', ..."
)
    return cols

def _cand_dyn_cols(inst: str):
    return [f"{inst}_dynamic", f"{inst}_dyn", f"dynamic_{inst}", f"dyn_{inst}"]

def _cand_tec_cols(inst: str):
    return [f"{inst}_technique", f"{inst}_tec", f"technique_{inst}", f"tec_{inst}"]

# ===================== CSV inventory =====================
_CSV_MAP = {
    "duos_loose": ("harmonic_intervals_loose", [
        "cello_viola_loose.csv", "cello_violin_loose.csv", "viola_violin_loose.csv"]),
    "trios_loose": ("triads_loose", [
        "triads_major_loose.csv", "triads_minor_loose.csv",
        "triads_diminished_loose.csv", "triads_augmented_loose.csv"]),
    "quartets_loose": ("7th_chords_loose", [
        "major_seventh_loose.csv", "minor_seventh_loose.csv",
        "dominant_seventh_loose.csv", "half_diminished_seventh_loose.csv"]),
    "duos_strict": ("harmonic_intervals_strict", [
        "cello_viola_strict.csv", "cello_violin_strict.csv", "viola_violin_strict.csv"]),
    "trios_strict": ("triads_strict", [
        "triads_major_strict.csv", "triads_minor_strict.csv",
        "triads_diminished_strict.csv", "triads_augmented_strict.csv"]),
    "quartets_strict": ("7th_chords_strict", [
        "major_seventh_strict.csv", "minor_seventh_strict.csv",
        "dominant_seventh_strict.csv", "half_diminished_seventh_strict.csv"]),
}

def _build_csv_list(group_key):
    folder, files = _CSV_MAP[group_key]
    return [os.path.join(paths.base, folder, "csv", f) for f in files]

csv_duos_loose  = _build_csv_list("duos_loose")
csv_trios_loose  = _build_csv_list("trios_loose")
csv_quartets_loose = _build_csv_list("quartets_loose")
csv_duos_strict  = _build_csv_list("duos_strict")
csv_trios_strict = _build_csv_list("trios_strict")
csv_quartets_strict = _build_csv_list("quartets_strict")

ALL_CSVS = (
    csv_duos_loose + csv_trios_loose + csv_quartets_loose +
    csv_duos_strict + csv_trios_strict + csv_quartets_strict
)

# ===================== Fail-fast inventory check =====================
_missing = [p for p in ALL_CSVS if not os.path.isfile(p)]
if _missing:
    print("Missing CSV files:")
    for p in _missing: print(" -", p)
    raise FileNotFoundError(f"{len(_missing)} CSV(s) missing — fix paths and re-run this cell.")

print(f" CSV inventory OK — {len(ALL_CSVS)} files found.")
print(f" DYNAMICS : {len(DYNAMICS)} classes")
print(f" TECHNIQUES : {len(TECHNIQUES)} classes")
print(f" CACHE_ROOT : {CACHE_ROOT}")
print(f" OUTDIR  : {paths.outdir}")


## Cell 4 — Load Metadata & Apply Arco-Normal Constraint (EXP3)
Loads SECD metadata and keeps only samples where all present instruments use
`arco-normal` technique. This defines the global dynamics classification
dataset used by EXP3.


In [ ]:
# Cell 4 — Load metadata & apply arco-normal constraint

# --- Safety: ensure helpers from Cell 3 exist in this runtime ---
try:
    _ = (resolve_filename_column, derive_audio_root_from_csv, _cand_dyn_cols, _cand_tec_cols)
except NameError:
    CANDIDATE_FILENAME_COLS = ["chord_filename", "filename", "wav_filename", "output_wav", "wav_file"]

    def resolve_filename_column(df: pd.DataFrame) -> str:
        for c in CANDIDATE_FILENAME_COLS:
            if c in df.columns:
                return c
        raise KeyError(f"Filename column not found in CSV. Tried: {CANDIDATE_FILENAME_COLS}")

    def derive_audio_root_from_csv(csv_path: str) -> str:
        csv_dir, csv_file = os.path.split(csv_path)
        base = os.path.splitext(csv_file)[0]
        wav_dir = csv_dir.replace("/csv", "/wav")
        return os.path.join(wav_dir, base) + "/"

    def _cand_dyn_cols(inst: str):
        return [f"{inst}_dynamic", f"{inst}_dyn", f"dynamic_{inst}", f"dyn_{inst}"]

    def _cand_tec_cols(inst: str):
        return [f"{inst}_technique", f"{inst}_tec", f"technique_{inst}", f"tec_{inst}"]

def _abs_path(audio_root: str, x: str) -> str:
    x = str(x).strip()
    return x if os.path.isabs(x) else os.path.join(audio_root, x)


def _resolve_audio_path(audio_root: str, filename: str) -> str:
    raw = str(filename).strip()
    if os.path.isabs(raw) and os.path.exists(raw):
        return raw
    candidates = []
    if os.path.isabs(raw):
        candidates.append(raw)
    else:
        candidates.append(os.path.join(audio_root, raw))

    stem = os.path.splitext(os.path.basename(raw))[0]
    candidates.append(os.path.join(audio_root, stem + ".wav"))

    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[-1]

# def _cache_path(cache_root: str, domain_tag: str, ensemble_tag: str, abs_fp: str) -> str:
#  stem = os.path.splitext(os.path.basename(abs_fp))[0]
#  name = f"{domain_tag}_{ensemble_tag}_{stem}.npy"
#  return os.path.join(cache_root, name)

def _cache_path(cache_root, csv_path, wav_filename):
    """Derive cache path from CSV path structure + wav filename."""
    csv_dir = os.path.dirname(csv_path)     # .../harmonic_intervals_loose/csv
    group = os.path.basename(os.path.dirname(csv_dir))  # harmonic_intervals_loose
    subset = os.path.splitext(os.path.basename(csv_path))[0] # cello_viola_loose
    stem = os.path.splitext(os.path.basename(str(wav_filename).strip()))[0]
    return os.path.join(cache_root, group, subset, f"{stem}.npy")

def _get_cell(dfrow, names):
    for c in names:
        if c in dfrow.index and pd.notna(dfrow[c]):
            return str(dfrow[c]).strip()
    return None

def _strict_agree(vals):
    vals = [v for v in vals if v is not None]
    return len(vals) > 0 and len(set(vals)) == 1

def load_block(csv_list, ensemble_tag, domain_tag, cache_root):
    frames = []
    voices = {"duo": 2, "trio": 3, "quartet": 4}[ensemble_tag]

    for path in csv_list:
        if not os.path.exists(path):
            print(f" Warning: CSV not found, skipping: {path}")
            continue

        df = pd.read_csv(path)
        fname_col = resolve_filename_column(df)
        audio_root = derive_audio_root_from_csv(path)

        std = {
            "dyn_cello": [], "tec_cello": [],
            "dyn_viola": [], "tec_viola": [],
            "dyn_violin1": [], "tec_violin1": [],
            "dyn_violin2": [], "tec_violin2": [],
            "is_strict_labels": []
        }

        for _, r in df.iterrows():
            d_global = _get_cell(r, ["dynamic"])
            t_global = _get_cell(r, ["technique"])

            d_cel = _get_cell(r, _cand_dyn_cols("cello"))
            t_cel = _get_cell(r, _cand_tec_cols("cello"))

            d_vla = _get_cell(r, _cand_dyn_cols("viola"))
            t_vla = _get_cell(r, _cand_tec_cols("viola"))

            d_v1 = _get_cell(r, _cand_dyn_cols("violin1"))
            t_v1 = _get_cell(r, _cand_tec_cols("violin1"))

            d_v2 = _get_cell(r, _cand_dyn_cols("violin2"))
            t_v2 = _get_cell(r, _cand_tec_cols("violin2"))

            if d_global is not None:
                if d_cel is None:
                    d_cel = d_global
                if d_vla is None:
                    d_vla = d_global
                if d_v1 is None:
                    d_v1 = d_global

            if t_global is not None:
                if t_cel is None:
                    t_cel = t_global
                if t_vla is None:
                    t_vla = t_global
                if t_v1 is None:
                    t_v1 = t_global

            std["dyn_cello"].append(d_cel)
            std["tec_cello"].append(t_cel)
            std["dyn_viola"].append(d_vla)
            std["tec_viola"].append(t_vla)
            std["dyn_violin1"].append(d_v1)
            std["tec_violin1"].append(t_v1)
            std["dyn_violin2"].append(d_v2)
            std["tec_violin2"].append(t_v2)

            dyn_present = [x for x in [d_cel, d_vla, d_v1, d_v2] if x is not None]
            tec_present = [x for x in [t_cel, t_vla, t_v1, t_v2] if x is not None]

            std["is_strict_labels"].append(
                _strict_agree(dyn_present) and _strict_agree(tec_present)
)

        df_std = pd.DataFrame(std)

        df["subset"] = domain_tag
        df["ensemble_size"] = ensemble_tag
        df["voices"] = voices

        df["filepath"] = df[fname_col].apply(lambda x: _resolve_audio_path(audio_root, x))
        df["cachefile"] = df[fname_col].apply(
            lambda x: _cache_path(CACHE_ROOT, path, x)
)

        df = pd.concat([df.reset_index(drop=True), df_std.reset_index(drop=True)], axis=1)

        df = df[df["filepath"].apply(os.path.exists())].reset_index(drop=True)

        if len(df) == 0:
            print(f" Warning: empty block after filtering for {path}")

        frames.append(df)

    return frames

all_dfs = []

all_dfs += load_block(csv_duos_loose, "duo", "loose", CACHE_ROOT)
all_dfs += load_block(csv_trios_loose, "trio", "loose", CACHE_ROOT)
all_dfs += load_block(csv_quartets_loose, "quartet", "loose", CACHE_ROOT)

all_dfs += load_block(csv_duos_strict, "duo", "strict", CACHE_ROOT)
all_dfs += load_block(csv_trios_strict, "trio", "strict", CACHE_ROOT)
all_dfs += load_block(csv_quartets_strict, "quartet", "strict", CACHE_ROOT)

if not all_dfs:
    raise FileNotFoundError("No valid data was loaded.")

df_all = pd.concat(all_dfs, ignore_index=True)

print(f"Loaded {len(df_all)} total samples (before filtering)")

for inst in INSTRUMENTS:
    df_all[f"has_{inst}"] = df_all[f"dyn_{inst}"].notna().astype(int)

def _present_signature(row):
    return tuple([inst for inst in INSTRUMENTS if row[f"has_{inst}"] == 1])

df_all["present_signature"] = df_all.apply(_present_signature, axis=1)

def _is_arco_normal(row):
    valid = []

    for inst in INSTRUMENTS:
        val = row.get(f"tec_{inst}", None)

        if pd.isna(val):
            continue

        val = str(val).strip()

        if val == "" or val.lower() == "nan":
            continue

        valid.append(val)

    if len(valid) == 0:
        return True

    return all(v == "arco-normal" for v in valid)

before = len(df_all)
df_all = df_all[df_all.apply(_is_arco_normal, axis=1)].reset_index(drop=True)

print(f"\nfilter: {before} → {len(df_all)} samples")

print("\nSubset × Ensemble:")
print(pd.crosstab(df_all["subset"], df_all["ensemble_size"]))

print("\nEnsemble counts:")
print(df_all["ensemble_size"].value_counts())


## Cell 5 — Stratified Train / Val / Test Split (70 / 15 / 15)
Stratifies by dynamics-related multi-label structure to preserve label balance across splits. Saves indices to disk.


In [ ]:
# Cell 5 — Stratified Train / Val / Test Split (70 / 15 / 15) — 

OUT_SPLIT_DIR = Path(paths.split_dir)
OUT_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

_split_file = OUT_SPLIT_DIR / "exp3_split_indices.npz"
_manifest_file = OUT_SPLIT_DIR / "exp3_manifest.csv"

# ===================== CACHE VALIDATION =====================
use_cache = False

if _split_file.exists() and _manifest_file.exists():
    manifest = pd.read_csv(_manifest_file)

    if len(manifest) == len(df_all):
        use_cache = True
    else:
        print("Cached splits incompatible with current dataset — recomputing...")
        _split_file.unlink(missing_ok=True)
        _manifest_file.unlink(missing_ok=True)

# ===================== LOAD OR COMPUTE =====================
if use_cache:
    print("Loading existing splits from disk...")

    _npz  = np.load(_split_file)
    train_idx = _npz["train_idx"]
    val_idx = _npz["val_idx"]
    test_idx = _npz["test_idx"]

    total = len(train_idx) + len(val_idx) + len(test_idx)

    print(f"Split sizes — train: {len(train_idx)} ({len(train_idx)/total:.1%}) "
            f"| val: {len(val_idx)} ({len(val_idx)/total:.1%}) "
            f"| test: {len(test_idx)} ({len(test_idx)/total:.1%})")

else:
    print("Computing splits...")

    dyn_values = sorted(pd.unique(
        pd.concat([df_all[f"dyn_{inst}"] for inst in INSTRUMENTS], ignore_index=True)
            .dropna().astype(str)
))

    blocks = []

    for inst in INSTRUMENTS:
        dcat = pd.Categorical(df_all[f"dyn_{inst}"].astype("string"), categories=dyn_values)
        dOH = pd.get_dummies(dcat).values.astype(np.int8)
        blocks.append(sparse.csr_matrix(dOH))

    ens_categories = sorted(df_all["ensemble_size"].astype(str).unique())
    ens_cat = pd.Categorical(df_all["ensemble_size"].astype(str), categories=ens_categories)
    ens_OH = pd.get_dummies(ens_cat).values.astype(np.int8)

    blocks.append(sparse.csr_matrix(ens_OH))

    Y = sparse.hstack(blocks, format="csr", dtype=np.int8)

    print("Strat matrix:", Y.shape)

    def iterative_split(n_items, Y, seed=1337):
        X = np.arange(n_items)
        rng = np.random.RandomState(seed)
        perm = rng.permutation(n_items)
        Xp, Yp = X[perm], Y[perm]

        s1 = IterativeStratification(
            n_splits=2, order=2,
            sample_distribution_per_fold=[0.3, 0.7]
)
        fold_a, fold_b = next(s1.split(Xp, Yp))
        tr_rel, tmp_rel = fold_a, fold_b

        s2 = IterativeStratification(
            n_splits=2, order=2,
            sample_distribution_per_fold=[0.5, 0.5]
)
        fold_c, fold_d = next(s2.split(Xp[tmp_rel], Yp[tmp_rel]))
        va_rel, te_rel = fold_c, fold_d

        return Xp[tr_rel], Xp[tmp_rel][va_rel], Xp[tmp_rel][te_rel]

    idx_strict = np.where(df_all["subset"].values == "strict")[0]
    idx_loose = np.where(df_all["subset"].values == "loose")[0]

    tr_s, va_s, te_s = iterative_split(len(idx_strict), Y[idx_strict])
    tr_l, va_l, te_l = iterative_split(len(idx_loose), Y[idx_loose])

    train_idx = np.concatenate([idx_strict[tr_s], idx_loose[tr_l]])
    val_idx = np.concatenate([idx_strict[va_s], idx_loose[va_l]])
    test_idx = np.concatenate([idx_strict[te_s], idx_loose[te_l]])

    rng = np.random.RandomState(1337)
    for arr in (train_idx, val_idx, test_idx):
        rng.shuffle(arr)

    total = len(train_idx) + len(val_idx) + len(test_idx)

    print(f"Split sizes — train: {len(train_idx)} ({len(train_idx)/total:.1%}) "
            f"| val: {len(val_idx)} ({len(val_idx)/total:.1%}) "
            f"| test: {len(test_idx)} ({len(test_idx)/total:.1%})")

    S_tr, S_va, S_te = set(train_idx), set(val_idx), set(test_idx)

    assert len(S_tr & S_va) == 0
    assert len(S_tr & S_te) == 0
    assert len(S_va & S_te) == 0
    assert len(S_tr | S_va | S_te) == len(df_all)

    print("Disjointness & coverage: OK")

    np.savez_compressed(
        _split_file,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx
)

    manifest = pd.DataFrame({
        "idx": np.arange(len(df_all)),
        "split": "none",
        "subset": df_all["subset"].values,
        "ensemble_size": df_all["ensemble_size"].values,
    })

    manifest.loc[train_idx, "split"] = "train"
    manifest.loc[val_idx, "split"] = "val"
    manifest.loc[test_idx, "split"] = "test"

    manifest.to_csv(_manifest_file, index=False)

    print("Saved splits")

# ===================== REPORT =====================
for name, idx in [("TRAIN", train_idx), ("VAL", val_idx), ("TEST", test_idx)]:
    print(f"\n{name} — subset x ensemble_size")
    print(pd.crosstab(df_all.iloc[idx]["subset"], df_all.iloc[idx]["ensemble_size"]))


## Cell 6b — Reload Splits from Disk
Run instead of Cell 5 on every session restart.


In [ ]:
# Cell 6b — Reload splits from disk (run instead of Cell 5 on every session restart)

if "df_all" not in globals():
    raise RuntimeError(
        "df_all is missing. Re-run Cells 3 → 4 first."
)

OUT_SPLIT_DIR = Path(paths.split_dir)
NPZ_PATH  = OUT_SPLIT_DIR / "exp3_split_indices.npz"
CSV_PATH  = OUT_SPLIT_DIR / "exp3_manifest.csv"

for p in [NPZ_PATH, CSV_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Split file not found: {p}\nRun Cell 5 once to generate it."
)

manifest = pd.read_csv(CSV_PATH)
if len(manifest) != len(df_all):
    raise RuntimeError(
        "Split files are incompatible with current df_all. Re-run Cell 5."
)

data  = np.load(NPZ_PATH)
train_idx = data["train_idx"]
val_idx = data["val_idx"]
test_idx = data["test_idx"]

df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_val = df_all.iloc[val_idx].reset_index(drop=True)
df_test = df_all.iloc[test_idx].reset_index(drop=True)

total = len(train_idx) + len(val_idx) + len(test_idx)

print(f"Splits reloaded — "
        f"train: {len(df_train)} ({len(df_train)/total:.1%}) | "
        f"val: {len(df_val)} ({len(df_val)/total:.1%}) | "
        f"test: {len(df_test)} ({len(df_test)/total:.1%})")

s_tr = set(train_idx.tolist())
s_va = set(val_idx.tolist())
s_te = set(test_idx.tolist())

assert s_tr.isdisjoint(s_va)
assert s_tr.isdisjoint(s_te)
assert s_va.isdisjoint(s_te)

print("Disjointness: OK")

for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    subsets = df["subset"].unique().tolist()
    assert "strict" in subsets and "loose" in subsets, \
        f"{name} split is missing a subset — re-run Cell 5."

print("Subset coverage: OK")

for name, df in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    print(f"\n{name} — subset x ensemble_size")
    print(pd.crosstab(df["subset"], df["ensemble_size"]))


## Cell 11.8 — Focus Dynamics Selection (TRAIN only)
Defines `FOCUS_DYN` — the 5 dynamics classes used for classification.


In [ ]:
# Cell 11.8 — Focus dynamics selection (TRAIN only)

if "df_train" not in globals():
    raise RuntimeError("df_train not found. Run Cells 3 → 4 → 5/6b first.")

def _collect_dyn_labels(df: pd.DataFrame) -> pd.Series:
    cols = [f"dyn_{inst}" for inst in INSTR_ORDER if f"dyn_{inst}" in df.columns]
    if not cols:
        return pd.Series(dtype=object)
    return pd.concat([df[c].dropna().astype(str) for c in cols], ignore_index=True)

dyn_counts = _collect_dyn_labels(df_train).value_counts()

FOCUS_DYN = [
    "pianissimo",
    "piano",
    "mezzo-piano",
    "forte",
    "fortissimo",
]

missing_from_train = [x for x in FOCUS_DYN if x not in dyn_counts.index]
if missing_from_train:
    raise RuntimeError(f"Selected focus dynamics missing from training data: {missing_from_train}")

focus_coverage = dyn_counts[dyn_counts.index.isin(FOCUS_DYN)].sum()
total_dyn = dyn_counts.sum()

print(f"FOCUS_DYN: {FOCUS_DYN}")
print(f"TRAIN dynamics coverage: {int(focus_coverage):,} / {int(total_dyn):,} ({100.0 * focus_coverage / total_dyn:.1f}%)")

DYN_FREQ_TRAIN = dyn_counts

os.makedirs(paths.split_dir, exist_ok=True)

with open(os.path.join(paths.split_dir, "focus_labels.json"), "w") as f:
    json.dump({"dyn": FOCUS_DYN}, f, indent=2)

print("Saved →", os.path.join(paths.split_dir, "focus_labels.json"))


## Cell 11.9 — Focus Label Filtering
Keeps only samples whose present-instrument dynamics belong to `FOCUS_DYN`.


In [ ]:
# Cell 11.9 — Focus label filtering

assert "FOCUS_DYN" in globals(), "Run Cell 11.8 first."
assert all(x in globals() for x in ["df_train", "df_val", "df_test"]), \
    "Run Cells 3 → 4 → 6b first."

def _row_in_focus_dyn(row):
    vals = []
    for inst in INSTR_ORDER:
        col = f"dyn_{inst}"
        if col in row and pd.notna(row[col]):
            vals.append(str(row[col]))

    if not vals:
        return False

    return all(v in FOCUS_DYN for v in vals)

before_tr = len(df_train)
before_va = len(df_val)
before_te = len(df_test)

df_train = df_train[df_train.apply(_row_in_focus_dyn, axis=1)].reset_index(drop=True)
df_val = df_val[df_val.apply(_row_in_focus_dyn, axis=1)].reset_index(drop=True)
df_test = df_test[df_test.apply(_row_in_focus_dyn, axis=1)].reset_index(drop=True)

print("Filtering to FOCUS_DYN only:")
print(f" train: {before_tr:,} → {len(df_train):,} ({len(df_train)/before_tr:.1%})")
print(f" val : {before_va:,} → {len(df_val):,} ({len(df_val)/before_va:.1%})")
print(f" test : {before_te:,} → {len(df_test):,} ({len(df_test)/before_te:.1%})")

def _check(df, name):
    bad = 0
    for _, r in df.iterrows():
        for inst in INSTR_ORDER:
            col = f"dyn_{inst}"
            if col in df.columns and pd.notna(r[col]):
                if str(r[col]) not in FOCUS_DYN:
                    bad += 1
                    break
    assert bad == 0, f"{name}: Found non-focus labels!"

_check(df_train, "train")
_check(df_val, "val")
_check(df_test, "test")

print("Focus label filtering: OK")


## Cell 11.10 — Class Weights for Imbalance Correction
Computes effective-number class weights for the 5 dynamics classes.


In [ ]:
# Cell 11.10 — Class weights for imbalance (effective number; dynamics only)

assert "FOCUS_DYN" in globals(), \
    "Run Cell 11.8 first (to compute focus labels)."
assert isinstance(df_train, pd.DataFrame), \
    "df_train missing (run earlier split cells)."

def _counts_from_df(df, focus):
    counts = {k: 0 for k in focus}
    for inst in INSTR_ORDER:
        col = f"dyn_{inst}"
        if col not in df.columns:
            continue
        vc = df[col].dropna().astype(str).value_counts()
        for k in focus:
            counts[k] += int(vc.get(k, 0))
    return counts

def _class_weights_from_counts(counts_dict, beta=0.9999):
    labels = list(counts_dict.keys())
    ns = np.array([max(1, counts_dict[k]) for k in labels], dtype=np.float64)
    w = (1.0 - beta) / (1.0 - np.power(beta, ns))
    w = w / w.mean()
    return labels, w.astype(np.float32)

dyn_counts = _counts_from_df(df_train, FOCUS_DYN)
dyn_keys, dyn_w = _class_weights_from_counts(dyn_counts, beta=0.9999)

assert dyn_keys == FOCUS_DYN, \
    "Weight order mismatch with FOCUS_DYN!"

CLASS_WEIGHTS_DYN = torch.tensor(dyn_w, dtype=torch.float32)

print("DYN weights (mean≈1):", np.round(CLASS_WEIGHTS_DYN.numpy(), 3))
print("Counts:", dyn_counts)

os.makedirs(paths.split_dir, exist_ok=True)

np.save(os.path.join(paths.split_dir, "class_weights_dyn.npy"), CLASS_WEIGHTS_DYN.numpy())

with open(os.path.join(paths.split_dir, "class_weights_counts.json"), "w") as f:
    json.dump({"dyn_counts": dyn_counts}, f, indent=2)

print("Saved class weights →", paths.split_dir)


## Cell 11.11 — Dataset, Collate, and Split Construction (EXP3)
Defines `SECDDataset` that returns `input_values` (128×256 mel) and
`y_dyn` (per-instrument dynamics labels). This is trained as a masked
multi-target dynamics prediction task.


In [ ]:
# Cell 11.11 — Dataset, collate, split construction (EXP3)

for _n in ("df_train", "df_val", "df_test", "FOCUS_DYN", "INSTR_ORDER"):
    if _n not in globals():
        raise RuntimeError(f"{_n} missing — run previous cells.")

# AST-native: 128 mel bins x 256 frames
TARGET_FRAMES = 256

def pad_or_truncate(mel, n_frames):
    t = mel.shape[-1]
    if t >= n_frames:
        return mel[..., :n_frames]
    return torch.nn.functional.pad(mel, (0, n_frames - t))

dyn_labels = FOCUS_DYN
dyn2i = {s: i for i, s in enumerate(dyn_labels)}
N_DYN = len(dyn_labels)

print("Dynamics classes:", dyn_labels)

class SECDDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _lab(self, row, inst):
        col = f"dyn_{inst}"
        if col not in row or row[col] is None:
            return -1
        return dyn2i.get(str(row[col]), -1)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            mel = torch.tensor(np.load(row["cachefile"]).astype(np.float32))
        except:
            mel = torch.zeros(128, TARGET_FRAMES)

        mel = pad_or_truncate(mel, TARGET_FRAMES)

        y_dyn = torch.tensor(
            [self._lab(row, inst) for inst in INSTR_ORDER],
            dtype=torch.long
)

        return {
            "input_values": mel,
            "y_dyn": y_dyn,
        }

def collate_fn(batch):
    return {
        "input_values": torch.stack([b["input_values"] for b in batch]),
        "y_dyn": torch.stack([b["y_dyn"] for b in batch]),
    }

ds_train = SECDDataset(df_train)
ds_val = SECDDataset(df_val)
ds_test = SECDDataset(df_test)

print("Train:", len(ds_train))
print("Val :", len(ds_val))
print("Test :", len(ds_test))


## Cell 12 — Full Training Pipeline (EXP3 / dynamics classification)
Defines and trains the AST-based dynamics model with masked loss for per-instrument
dynamics targets .
**This cell is fully self-contained** — `resize_ast_pos_embed` and `masked_ce` are defined inline.


In [ ]:
# Cell 12 — Full Training Pipeline (EXP3 / dynamics classification)

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

need = [
    "INSTR_ORDER", "TARGET_FRAMES", "FOCUS_DYN", "CLASS_WEIGHTS_DYN",
    "ds_train", "ds_val", "ds_test", "collate_fn"
]
miss = [k for k in need if k not in globals()]
if miss:
    raise RuntimeError(f"Missing from previous cells: {miss}")

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found.")

print(f"GPU: {torch.cuda.get_device_name(0)}")

BASE = paths.outdir

MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
SEED = 1337

torch.manual_seed(SEED)
np.random.seed(SEED)

stamp = time.strftime("%Y%m%d-%H%M%S")
OUTDIR = os.path.join(BASE, f"exp3_ast_dyn_full_{stamp}")
os.makedirs(OUTDIR, exist_ok=True)

with open(os.path.join(BASE, "latest_run.txt"), "w") as f:
    f.write(OUTDIR)

with open(os.path.join(OUTDIR, "session_config.json"), "w") as f:
    json.dump({
        "MODEL_NAME": MODEL_NAME,
        "FOCUS_DYN": FOCUS_DYN,
        "TARGET_FRAMES": TARGET_FRAMES,
        "INSTR_ORDER": INSTR_ORDER,
        "CLASS_WEIGHTS_DYN": CLASS_WEIGHTS_DYN.detach().cpu().tolist(),
    }, f, indent=2)

def resize_ast_pos_embed(ast_model, target_frames):
    """Interpolate AST positional embeddings to match target frame count."""
    n_mels_patches = (128 - 16) // 10 + 1  # = 12
    n_time_patches = (target_frames - 16) // 10 + 1
    n_patches = n_mels_patches * n_time_patches
    n_tokens = n_patches + 2 # CLS + distillation

    pos_embed = ast_model.embeddings.position_embeddings
    old_n = pos_embed.shape[1]

    if old_n == n_tokens:
        return

    cls_dist = pos_embed[:, :2, :]
    patch_pos = pos_embed[:, 2:, :]

    old_freq = 12
    old_time = 101
    d = patch_pos.shape[-1]

    patch_pos = patch_pos.reshape(1, old_freq, old_time, d).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(
        patch_pos.float(),
        size=(n_mels_patches, n_time_patches),
        mode="bicubic",
        align_corners=False,
)
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, n_patches, d)

    new_pos = torch.cat([cls_dist, patch_pos], dim=1)
    ast_model.embeddings.position_embeddings = torch.nn.Parameter(new_pos)
    print(f" Pos embeddings resized: {old_n} → {n_tokens} "
            f"(grid {old_freq}×{old_time} → {n_mels_patches}×{n_time_patches})")

def compute_metrics_dyn(eval_pred):
    preds, labels = eval_pred
    logits = np.asarray(preds)
    y_true = np.asarray(labels)

    y_pred = logits.argmax(axis=-1).reshape(-1)
    y_true = y_true.reshape(-1)

    mask = y_true >= 0
    yt = y_true[mask]
    yp = y_pred[mask]

    return {
        "eval_support_dyn": int(mask.sum()),
        "eval_acc_dyn": float(accuracy_score(yt, yp)),
        "eval_f1_macro_dyn": float(f1_score(yt, yp, average="macro", zero_division=0)),
    }

def masked_ce(logits, targets, weight=None, ignore_index=-1):
    valid = targets.ne(ignore_index)
    if not valid.any():
        return logits.new_zeros(())
    return F.cross_entropy(
        logits[valid].float(),
        targets[valid],
        weight=weight.float() if weight is not None else None,
        label_smoothing=0.05,
)

class ASTMultiHeadDynamics(nn.Module):
    def __init__(self, model_name, instr_order, n_dyn, class_weights_dyn):
        super().__init__()
        self.n_dyn = n_dyn
        self.n_instr = len(instr_order)

        self.ast = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        resize_ast_pos_embed(self.ast, TARGET_FRAMES)

        with torch.no_grad():
            dummy = torch.zeros(1, 128, TARGET_FRAMES)
            h = self.ast(input_values=dummy).last_hidden_state
            d = h.mean(dim=1).shape[-1]

        self.inst_embed = nn.Embedding(self.n_instr, d)

        self.heads_dyn = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, d),
                nn.GELU(),
                nn.Linear(d, n_dyn)
)
            for _ in range(self.n_instr)
        ])

        self.register_buffer("w_dyn", torch.tensor(class_weights_dyn, dtype=torch.float32))

    def feats(self, x):
        return self.ast(input_values=x).last_hidden_state.mean(dim=1)

    def forward(self, input_values, y_dyn=None, **kwargs):
        emb = self.feats(input_values)

        inst_ids = torch.arange(self.n_instr, device=emb.device)
        inst_vecs = self.inst_embed(inst_ids)

        logits_list = []
        for i, head in enumerate(self.heads_dyn):
            inst_vec = inst_vecs[i].unsqueeze(0).expand_as(emb)
            conditioned = emb + inst_vec
            logits_list.append(head(conditioned))

        logits = torch.stack(logits_list, dim=1)

        loss = None
        if y_dyn is not None:
            loss = masked_ce(
                logits.reshape(-1, self.n_dyn),
                y_dyn.reshape(-1),
                weight=self.w_dyn
)

        return {"loss": loss, "logits": logits}

class EpochTableCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.train_losses.append(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        if state.epoch == 1:
            print("-"*80)
            print("Epoch | Train Loss | Val Loss | Acc | F1")
            print("-"*80)

        ep = int(state.epoch or 0)

        tl = np.mean(self.train_losses) if self.train_losses else 0.0
        self.train_losses = []

        vl = metrics.get("eval_loss", 0.0)
        acc = metrics.get("eval_acc_dyn", 0.0)
        f1 = metrics.get("eval_f1_macro_dyn", 0.0)

        print(f"{ep:>5} | {tl:.4f} | {vl:.4f} | {acc:.4f} | {f1:.4f}")

device = DEVICE

model = ASTMultiHeadDynamics(
    MODEL_NAME,
    INSTR_ORDER,
    len(FOCUS_DYN),
    CLASS_WEIGHTS_DYN
).to(device)

model.ast.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=os.path.join(OUTDIR, "hf_ckpt"),
    logging_dir=os.path.join(OUTDIR, "logs"),

    num_train_epochs=DEMO_NUM_TRAIN_EPOCHS,
    max_steps=DEMO_MAX_STEPS,
    per_device_train_batch_size=DEMO_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=DEMO_EVAL_BATCH_SIZE,

    learning_rate=3e-5,
    warmup_ratio=0.05,
    weight_decay=1e-2,
    fp16=(DEVICE.type == "cuda"),
    dataloader_num_workers=DEMO_NUM_WORKERS,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro_dyn",
    greater_is_better=True,
    save_total_limit=2,

    remove_unused_columns=False,
    label_names=["y_dyn"],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collate_fn,
    compute_metrics=compute_metrics_dyn,
    callbacks=[EpochTableCallback(), EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer.remove_callback(PrinterCallback)

print("-"*72)
print(f"Run : {OUTDIR}")
print("-"*72)

t0 = time.time()
trainer.train()

print(f"\nTraining finished in {(time.time()-t0)/60:.2f} min")

trainer.save_model(os.path.join(args.output_dir, "final"))
trainer.save_state()

print("-"*72)
print("FULL RUN COMPLETE")
print(f"Artifacts -> {OUTDIR}")
print("-"*72)


## Cell 13 — Post-Training Evaluation and Artifacts (EXP3 dynamics task)
Loads the best checkpoint and runs inference on val/test sets.
**Self-contained** — `resize_ast_pos_embed` and `masked_ce` are defined inline.


In [ ]:
# Cell 13 — Post-Training Evaluation and Artifacts (EXP3 dynamics task)

matplotlib.use("Agg")

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BASE = paths.outdir

if "OUTDIR" not in globals() or not os.path.isdir(globals().get("OUTDIR", "")):
    ptr = os.path.join(BASE, "latest_run.txt")
    if not os.path.exists(ptr):
        raise RuntimeError("OUTDIR not found and latest_run.txt is missing.")
    with open(ptr, "r", encoding="utf-8") as f:
        OUTDIR = f.read.strip()
    print(f"Loaded run path -> {OUTDIR}")
else:
    print(f"Using OUTDIR -> {OUTDIR}")

CKPT_DIR = os.path.join(OUTDIR, "hf_ckpt")
ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

cfg_path = os.path.join(OUTDIR, "session_config.json")
if not os.path.exists(cfg_path):
    raise RuntimeError("Missing session_config.json")

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

FOCUS_DYN = cfg["FOCUS_DYN"]
TARGET_FRAMES = int(cfg["TARGET_FRAMES"])
INSTR_ORDER = list(cfg["INSTR_ORDER"])
MODEL_NAME = cfg["MODEL_NAME"]
W_DYN = np.array(cfg["CLASS_WEIGHTS_DYN"], dtype=np.float32)
N_DYN = len(FOCUS_DYN)

device = DEVICE
print(f"Device -> {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

need = ["ds_val", "ds_test", "collate_fn"]
for n in need:
    if n not in globals():
        raise RuntimeError(f"Missing: {n}")

ds_val_cap = ds_val
ds_test_cap = ds_test

print(f"VAL: {len(ds_val_cap):,} | TEST: {len(ds_test_cap):,}")

def resize_ast_pos_embed(ast_model, target_frames):
    n_mels_patches = (128 - 16) // 10 + 1
    n_time_patches = (target_frames - 16) // 10 + 1
    n_patches = n_mels_patches * n_time_patches
    n_tokens = n_patches + 2
    pos_embed = ast_model.embeddings.position_embeddings
    old_n = pos_embed.shape[1]
    if old_n == n_tokens:
        return
    cls_dist = pos_embed[:, :2, :]
    patch_pos = pos_embed[:, 2:, :]
    old_freq, old_time, d = 12, 101, patch_pos.shape[-1]
    patch_pos = patch_pos.reshape(1, old_freq, old_time, d).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(patch_pos.float(), size=(n_mels_patches, n_time_patches), mode="bicubic", align_corners=False)
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, n_patches, d)
    new_pos = torch.cat([cls_dist, patch_pos], dim=1)
    ast_model.embeddings.position_embeddings = torch.nn.Parameter(new_pos)

def masked_ce(logits, targets, weight=None, ignore_index=-1):
    valid = targets.ne(ignore_index)
    if not valid.any():
        return logits.new_zeros(())
    return torch.nn.functional.cross_entropy(
        logits[valid].float(),
        targets[valid],
        weight=weight.float() if weight is not None else None,
)

class ASTMultiHeadDynamics(nn.Module):
    def __init__(self, model_name, instr_order, n_dyn, class_weights_dyn):
        super().__init__()
        self.n_dyn = n_dyn
        self.n_instr = len(instr_order)
        self.ast = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        resize_ast_pos_embed(self.ast, TARGET_FRAMES)
        with torch.no_grad():
            dummy = torch.zeros(1, 128, TARGET_FRAMES)
            h = self.ast(input_values=dummy).last_hidden_state
            d = h.mean(dim=1).shape[-1]
        self.inst_embed = nn.Embedding(self.n_instr, d)
        self.heads_dyn = nn.ModuleList([
            nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, n_dyn))
            for _ in range(self.n_instr)
        ])
        self.register_buffer("w_dyn", torch.tensor(class_weights_dyn, dtype=torch.float32))

    def feats(self, x):
        return self.ast(input_values=x).last_hidden_state.mean(dim=1)

    def forward(self, input_values, y_dyn=None, **kwargs):
        emb = self.feats(input_values)
        inst_ids = torch.arange(self.n_instr, device=emb.device)
        inst_vecs = self.inst_embed(inst_ids)
        logits_list = []
        for i, head in enumerate(self.heads_dyn):
            inst_vec = inst_vecs[i].unsqueeze(0).expand_as(emb)
            logits_list.append(head(emb + inst_vec))
        logits = torch.stack(logits_list, dim=1)
        loss = None
        if y_dyn is not None:
            loss = masked_ce(logits.reshape(-1, self.n_dyn), y_dyn.reshape(-1), weight=self.w_dyn)
        return {"loss": loss, "logits": logits}

def resolve_best_checkpoint(ckpt_dir):
    state_path = os.path.join(ckpt_dir, "trainer_state.json")
    if not os.path.exists(state_path):
        return None
    with open(state_path, "r", encoding="utf-8") as f:
        state = json.load(f)
    best = state.get("best_model_checkpoint")
    if best and os.path.isdir(best):
        print(f"BEST checkpoint -> {best}")
        return best
    return None

model_dir = resolve_best_checkpoint(CKPT_DIR)

if model_dir is None:
    checkpoints = sorted(
        glob.glob(os.path.join(CKPT_DIR, "checkpoint-*")),
        key=lambda x: int(x.split("-")[-1])
)
    if not checkpoints:
        raise RuntimeError("No checkpoints found")
    model_dir = checkpoints[-1]
    print(f"Fallback -> {model_dir}")

model = ASTMultiHeadDynamics(MODEL_NAME, INSTR_ORDER, N_DYN, W_DYN)

state_dict = st.load_file(
    os.path.join(model_dir, "model.safetensors"),
    device=str(device)
)
load_info = model.load_state_dict(state_dict, strict=False)
if load_info.missing_keys:
    print(f"Missing keys : {load_info.missing_keys}")
if load_info.unexpected_keys:
    print(f"Unexpected keys : {load_info.unexpected_keys}")

model.to(device)
model.eval()

eval_trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=ARTIFACT_DIR,
        per_device_eval_batch_size=64,
        report_to=[],
        remove_unused_columns=False,
        label_names=["y_dyn"],
        use_cpu=(device.type == "cpu"),
        seed=1337,
),
    data_collator=collate_fn,
)

def run_predictions(dataset):
    pred = eval_trainer.predict(dataset)
    raw_logits = pred.predictions
    raw_labels = pred.label_ids
    logits = np.asarray(raw_logits[0]) if isinstance(raw_logits, tuple) else np.asarray(raw_logits)
    labels = np.asarray(raw_labels[0]) if isinstance(raw_labels, tuple) else np.asarray(raw_labels)

    pred_ids = logits.argmax(axis=-1)
    pooled_true = labels.reshape(-1)
    pooled_pred = pred_ids.reshape(-1)
    mask = pooled_true >= 0
    pooled_true = pooled_true[mask]
    pooled_pred = pooled_pred[mask]

    per_inst = {}
    for i, name in enumerate(INSTR_ORDER):
        y_t = labels[:, i]
        y_p = pred_ids[:, i]
        m = y_t >= 0
        per_inst[name] = (y_t[m], y_p[m])

    return pooled_true, pooled_pred, per_inst

def report(y_true, y_pred, name):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    print(f"{name} | acc={rep['accuracy']:.4f} | f1={rep['macro avg']['f1-score']:.4f}")
    return rep

def save_cm(y_true, y_pred, labels, title, path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title, fontsize=12, pad=10)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def save_cm_norm(y_true, y_pred, labels, title, path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))), normalize="true")
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    ax.set_title(title, fontsize=12, pad=10)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

all_metrics = {}

for split, ds in [("val", ds_val_cap), ("test", ds_test_cap)]:
    print(f"\n=== {split.upper()} ===")

    yt, yp, per_inst = run_predictions(ds)

    all_metrics[split] = {
        "pooled": report(yt, yp, f"{split}-pooled")
    }

    save_cm(yt, yp, FOCUS_DYN,
            f"{split.upper()} | Dynamics (pooled) — Absolute",
            os.path.join(ARTIFACT_DIR, f"cm_{split}_dyn_abs.png"))
    save_cm_norm(yt, yp, FOCUS_DYN,
                    f"{split.upper()} | Dynamics (pooled) — Normalized",
                    os.path.join(ARTIFACT_DIR, f"cm_{split}_dyn_norm.png"))

    for inst, (yti, ypi) in per_inst.items():
        all_metrics[split][inst] = report(yti, ypi, f"{split}-{inst}")

        save_cm(yti, ypi, FOCUS_DYN,
                f"{split.upper()} | Dynamics ({inst}) — Absolute",
                os.path.join(ARTIFACT_DIR, f"cm_{split}_dyn_{inst}_abs.png"))
        save_cm_norm(yti, ypi, FOCUS_DYN,
                        f"{split.upper()} | Dynamics ({inst}) — Normalized",
                        os.path.join(ARTIFACT_DIR, f"cm_{split}_dyn_{inst}_norm.png"))

with open(os.path.join(ARTIFACT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

pngs = sorted(f for f in os.listdir(ARTIFACT_DIR) if f.endswith(".png"))
print(f"\nArtifacts saved -> {ARTIFACT_DIR}")
print(f" metrics.json + {len(pngs)} PNGs:")
for p in pngs:
    print(f" {p}")


## Cell 13.05 — Training Curves (Loss + F1)

Reads `trainer_state.json` from the HF checkpoint directory and parses `log_history` to extract per-epoch train loss, validation loss, validation accuracy, and validation macro F1. Produces two PNGs saved to `OUTDIR/artifacts/`: `loss_curves.png` (train vs val loss) and `f1_curves.png` (val accuracy + macro F1 over epochs). The F1 plot is skipped gracefully if the log history contains neither `eval_acc_dyn` nor `eval_f1_macro_dyn` entries.


In [ ]:
# Cell 13.05 — Training Curves (Loss + F1)

matplotlib.use("Agg")

ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")
CKPT_DIR = os.path.join(OUTDIR, "hf_ckpt")

state_path = os.path.join(CKPT_DIR, "trainer_state.json")
if not os.path.exists(state_path):
    raise RuntimeError(f"trainer_state.json not found in {CKPT_DIR}")

with open(state_path, "r", encoding="utf-8") as f:
    state = json.load(f)

log = state["log_history"]

train_epochs, train_loss = [], []
val_epochs, val_loss, val_acc, val_f1 = [], [], [], []

for entry in log:
    e = entry.get("epoch")
    if e is None:
        continue
    if "loss" in entry and "eval_loss" not in entry:
        train_epochs.append(e)
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        val_epochs.append(e)
        val_loss.append(entry["eval_loss"])
        val_acc.append(entry.get("eval_acc_dyn", None))
        val_f1.append(entry.get("eval_f1_macro_dyn", None))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_epochs, train_loss, "o-", ms=3, label="Train loss")
ax.plot(val_epochs, val_loss, "s-", ms=3, label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, "loss_curves.png"), dpi=150)
plt.close(fig)
print("loss_curves.png")

has_acc = any(v is not None for v in val_acc)
has_f1 = any(v is not None for v in val_f1)

if has_acc or has_f1:
    fig, ax = plt.subplots(figsize=(9, 5))
    if has_acc:
        ax.plot(val_epochs, val_acc, "s-", ms=3, label="Val accuracy")
    if has_f1:
        ax.plot(val_epochs, val_f1, "^-", ms=3, label="Val macro F1")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.set_title("Validation Accuracy & Macro F1")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(ARTIFACT_DIR, "f1_curves.png"), dpi=150)
    plt.close(fig)
    print("f1_curves.png")
else:
    print("No eval_acc_dyn / eval_f1_macro_dyn found — f1_curves.png skipped")

print(f"\nSaved → {ARTIFACT_DIR}")


## Cell 13.1 — Artifact Preview (EXP3 dynamics task)

Displays all PNG artifacts produced by Cells 13 and 13.05 inline in the notebook using `IPython.display`. The images are shown in a fixed order: training curves, pooled confusion matrices, then per-instrument confusion matrices. Missing files are listed at the end rather than raising an error, so the cell can be run even if only a subset of plots was generated.


In [ ]:
# Cell 13.1 — Artifact Preview (EXP3 dynamics task)

ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")

if not os.path.isdir(ARTIFACT_DIR):
    raise RuntimeError(f"Artifacts directory not found: {ARTIFACT_DIR}")

CORE_PLOTS = [
    "loss_curves.png",
    "f1_curves.png",
]

POOLED_CM = [
    "cm_val_dyn_abs.png",
    "cm_val_dyn_norm.png",
    "cm_test_dyn_abs.png",
    "cm_test_dyn_norm.png",
]

INSTRUMENTS = ["cello", "viola", "violin2", "violin1"]

def build_instr_cm(split):
    return [
        f"cm_{split}_dyn_{inst}_{kind}.png"
        for inst in INSTRUMENTS
        for kind in ["abs", "norm"]
    ]

INSTR_CM = build_instr_cm("val") + build_instr_cm("test")

PNG_ORDER = CORE_PLOTS + POOLED_CM + INSTR_CM

print("\n" + "=" * 70)
print(" Artifact Preview — Dynamics (pooled + per-instrument)")
print("=" * 70)

missing = []

for fname in PNG_ORDER:
    fpath = os.path.join(ARTIFACT_DIR, fname)
    if os.path.isfile(fpath):
        print(f"\n-- {fname} --")
        display(IPImage(filename=fpath))
    else:
        missing.append(fname)

if missing:
    print("\n" + "-" * 70)
    print("Missing artifacts:")
    for m in missing:
        print(f" {m}")
